# TPI Análisis de Datos - Telco Customer Churn

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andyargg/tpi-analisis-de-datos/blob/main/notebooks/TPI_Analisis_de_Datos.ipynb)

Introducción al Análisis de Datos - UTN Avellaneda - Div. 143
Profe: Zotti, Enzo Martín
Integrantes: Matias Asenjo, Damian Bello, Andres Arguindegui y J. Zubini

En este trabajo usamos el dataset Telco Customer Churn para ver por qué los clientes de una empresa
de telefonía/internet se dan de baja.

## 1. Los datos

### 1.1 Librerías y carga

Usamos pandas y numpy para trabajar con los datos, y matplotlib y seaborn para los gráficos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)  # para que muestre todas las columnas

El csv lo leemos directo desde el repo de github, asi se puede correr en Colab sin subir nada.

In [ ]:
url = "https://raw.githubusercontent.com/andyargg/tpi-analisis-de-datos/main/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print("filas:", df.shape[0])
print("columnas:", df.shape[1])

### 1.2 Primer vistazo

In [ ]:
df.head()

In [ ]:
df.info()

### 1.3 Problema e hipótesis

La empresa está perdiendo clientes y queremos entender qué tienen en común los que se van, para que
puedan hacer algo para retenerlos antes de que cancelen.

La variable que nos interesa es **Churn** (Yes = se dio de baja, No = sigue siendo cliente). Veamos
cuántos hay de cada uno:

In [ ]:
print(df["Churn"].value_counts())
print()
print(df["Churn"].value_counts(normalize=True) * 100)

Se fue más o menos el 26% de los clientes, o sea 1 de cada 4. Hay muchos más "No" que "Yes", así que
cuando comparemos grupos vamos a usar porcentajes y no cantidades.

Hipótesis que queremos comprobar:

- H1: los que tienen contrato mes a mes se van más que los de 1 o 2 años (se pueden ir cuando quieren).
- H2: los clientes nuevos se van más que los que hace mucho que están.
- H3: los que pagan más por mes se van más.
- H4: los que pagan con cheque se van más que los que tienen débito automático.

### 1.4 Tipo de dataset y tipos de variables

El dataset es **estructurado** porque es una tabla: cada fila es un cliente y todas las filas tienen
las mismas columnas. No es semi-estructurado (tipo JSON) ni no estructurado (texto libre, imágenes, etc).

Ahora clasificamos cada variable (nominal, ordinal, discreta o continua) y lo comparamos con el tipo
que le puso pandas.

In [ ]:
tipos = {
    "customerID": "identificador",
    "gender": "cualitativa nominal",
    "SeniorCitizen": "cualitativa nominal",
    "Partner": "cualitativa nominal",
    "Dependents": "cualitativa nominal",
    "tenure": "cuantitativa discreta",
    "PhoneService": "cualitativa nominal",
    "MultipleLines": "cualitativa nominal",
    "InternetService": "cualitativa nominal",
    "OnlineSecurity": "cualitativa nominal",
    "OnlineBackup": "cualitativa nominal",
    "DeviceProtection": "cualitativa nominal",
    "TechSupport": "cualitativa nominal",
    "StreamingTV": "cualitativa nominal",
    "StreamingMovies": "cualitativa nominal",
    "Contract": "cualitativa ordinal",
    "PaperlessBilling": "cualitativa nominal",
    "PaymentMethod": "cualitativa nominal",
    "MonthlyCharges": "cuantitativa continua",
    "TotalCharges": "cuantitativa continua",
    "Churn": "cualitativa nominal",
}

tabla = pd.DataFrame({
    "tipo": tipos,
    "dtype pandas": df.dtypes.astype(str),
    "valores distintos": df.nunique(),
})
tabla

In [ ]:
tabla["tipo"].value_counts()

Lo que no coincide:

- **TotalCharges** es plata (continua) pero pandas la leyó como texto (object). Seguro tiene algún valor
  raro que no es número, eso lo vemos en la limpieza.
- **SeniorCitizen** pandas la toma como número (int64) porque viene con 0 y 1, pero en realidad es
  sí/no, es categórica.
- **Contract** tiene orden (mes a mes < 1 año < 2 años) pero para pandas es texto normal.
- **customerID** es solo un código, no sirve para analizar, después la sacamos.

El resto está bien: las categóricas quedaron como texto y tenure y MonthlyCharges como números.

## 2. Limpieza

### 2.1 Valores nulos y TotalCharges

Primero vemos si hay nulos:

In [ ]:
df.isnull().sum()

Pandas dice que no hay ninguno, pero TotalCharges venía como texto, así que buscamos celdas que
estén "vacías" pero con espacios:

In [ ]:
vacios = df["TotalCharges"].str.strip() == ""
print("celdas vacías en TotalCharges:", vacios.sum())

df[vacios][["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

Son 11 y todos tienen **tenure = 0**, o sea que son clientes que se acaban de dar de alta y todavía
no les facturaron nada. Por eso los completamos con 0 en vez de borrarlos o poner el promedio (el
promedio no tendría sentido, en realidad no pagaron nada todavía).

Convertimos la columna a número. Con `errors="coerce"` lo que no se puede convertir queda como NaN
y después lo llenamos con 0.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("nulos después de convertir:", df["TotalCharges"].isnull().sum())

df["TotalCharges"] = df["TotalCharges"].fillna(0)
print("nulos ahora:", df["TotalCharges"].isnull().sum())
print("tipo:", df["TotalCharges"].dtype)

### 2.2 Duplicados

In [ ]:
print("filas duplicadas:", df.duplicated().sum())
print("customerID repetidos:", df["customerID"].duplicated().sum())
print("filas iguales sin contar el ID:", df.drop(columns="customerID").duplicated().sum())

No hay filas duplicadas ni IDs repetidos. Hay 22 filas que son iguales si no miramos el ID, pero son
clientes distintos que justo tienen el mismo plan y los mismos datos, así que las dejamos.

### 2.3 Tipos de datos

SeniorCitizen viene con 0 y 1. La pasamos a "No"/"Yes" para que quede igual que las otras columnas
de sí/no (Partner, Dependents, etc) y no se confunda con un número.

In [ ]:
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
df["SeniorCitizen"].value_counts()

### 2.4 Columnas: eliminar, renombrar y crear

Sacamos customerID porque es solo un código y no sirve para el análisis. También renombramos
`gender` a `Gender` porque era la única columna en minúscula y así queda todo igual.

In [ ]:
df = df.drop(columns=["customerID"])
df = df.rename(columns={"gender": "Gender"})
df.columns

Columnas nuevas:

- **ChurnNum**: Churn como número (1 = se fue, 0 = no). Sirve para sacar la tasa de baja con un
  promedio.
- **TotalEstimado**: cargo mensual x meses de antigüedad. Es lo que "debería" haber pagado si siempre
  hubiera pagado lo mismo.
- **CantServicios**: cuántos servicios extra tiene contratados (seguridad, backup, soporte, streaming, etc).
- **GrupoAntiguedad**: tenure agrupada en rangos de años, para comparar clientes nuevos y viejos.

In [ ]:
df["ChurnNum"] = (df["Churn"] == "Yes").astype(int)

df["TotalEstimado"] = df["MonthlyCharges"] * df["tenure"]

servicios = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
             "TechSupport", "StreamingTV", "StreamingMovies"]
df["CantServicios"] = (df[servicios] == "Yes").sum(axis=1)

df["GrupoAntiguedad"] = pd.cut(df["tenure"], bins=[-1, 12, 24, 48, 72],
                               labels=["0-1 año", "1-2 años", "2-4 años", "4-6 años"])

df[["tenure", "MonthlyCharges", "TotalCharges", "TotalEstimado",
    "CantServicios", "GrupoAntiguedad", "Churn", "ChurnNum"]].head()

### 2.5 Outliers

Buscamos valores atípicos en las variables numéricas con boxplots y con el método del rango
intercuartílico (IQR). Un valor es outlier si está por debajo de Q1 - 1.5*IQR o por arriba de
Q3 + 1.5*IQR.

In [ ]:
numericas = ["tenure", "MonthlyCharges", "TotalCharges"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, numericas):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
for col in numericas:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    cant = ((df[col] < lim_inf) | (df[col] > lim_sup)).sum()
    print(f"{col}: Q1={q1:.2f}  Q3={q3:.2f}  límites=[{lim_inf:.2f}, {lim_sup:.2f}]  outliers={cant}")

No hay outliers en ninguna de las tres. TotalCharges tiene valores bastante altos (llega a más de
8000) pero no pasan el límite, y tiene sentido porque son clientes que hace muchos años que están.
Como son datos reales y no errores, no sacamos nada.

### 2.6 Guardar los datos limpios

In [ ]:
import os

os.makedirs("data/processed", exist_ok=True)
df.to_csv("data/processed/telco_limpio.csv", index=False)

print("guardado:", df.shape)
df.info()

Quedaron 7043 filas y 24 columnas (sacamos customerID y agregamos 4). Ya no hay nulos y TotalCharges
es numérica.

## 3. Variables numéricas

(pendiente)

## 4. Variables categóricas

(pendiente)

## 5. Segmentación y conclusiones

(pendiente)